# Day 51 Revision Summary — Cloud Deployment Basics

- **Config via environment (12-factor rule #3) replaces every hardcoded value** — `config.py`'s `Settings(BaseSettings)` reads, in priority order, real shell environment variables > a `.env` file > hardcoded class defaults, and is instantiated once at import time (`settings = Settings()`). This is what lets ONE built Docker image run correctly on Render, Fly.io, and a Heroku-style PaaS with three different sets of env vars, with zero rebuilds.
- **Cloud platforms assign `$PORT` at container START time, not image BUILD time** — a Dockerfile's exec-array `CMD ["uvicorn", ..., "--port", "8000"]` form never expands shell variables; only the shell form `CMD uvicorn ... --port ${PORT}` reads the environment when the container actually starts, which is the only form that can pick up a platform-assigned port.
- **Structured JSON logging to stdout, not a file** — `cloud_ready_api.py`'s `JsonFormatter` emits one JSON object per log line when `LOG_FORMAT=json`, because cloud log aggregators (CloudWatch, Cloud Logging, etc.) parse JSON lines into queryable fields, and a log file inside an ephemeral container is lost the instant that container is replaced.
- **Graceful shutdown turns SIGTERM into a real waiting period, not an instant kill** — uvicorn wires the SIGTERM every cloud platform sends before terminating a container to the FastAPI `lifespan`'s shutdown phase, which waits for in-flight requests and scheduled `BackgroundTasks` to finish before the process actually exits; verified live by sending SIGTERM mid-job and watching the job finish first.
- **A from-scratch cloud-readiness linter catches deploy-breaking patterns with zero server or cloud account** — `cloud_readiness_check.py` pattern-matches the `.py` source text for `localhost`/`127.0.0.1` binds, hardcoded ports in `run()` calls, a missing `/health` route, scattered `os.environ` reads, hardcoded secrets, and file-based logging — the same static-analysis method as Day 50's `dockerfile_lint.py`, applied to Python source instead of a Dockerfile.
- **One Docker image, three interchangeable orchestration configs** — `render.yaml`, `fly.toml`, and `Procfile` all build the identical Dockerfile; each config file's only job is telling its platform how to run that image and which env vars (like `REDIS_URL`) to inject at deploy time, never baking them into the image itself.

## Today's material (Day 51)

Checked all sources under the `day51` Drive folder:

- **`README.md`** — states no cloud account, credentials, or outbound access to cloud provider APIs is available in the class sandbox, and lays out the folder structure: `formulas.md`, `slides/`, `classwork/` (`config.py`, `cloud_ready_api.py`, `cloud_readiness_check.py`, `app_bad_example.py`, `redis_job_store.py` reused from Day 50, `Dockerfile`, `.dockerignore`, `Procfile`, `render.yaml`, `fly.toml`, `cloud_deploy_commands.md`, `.env.example`, reused CNN model + sample images), `homework/`, `resources/`.
- **Slides** (`slides/week11_day7_cloud_deployment_basics.html`) — 22-slide deck covering the same six sections as `formulas.md`: environment-based config, `$PORT` binding, structured logging, graceful shutdown, the cloud-readiness linter, and the three deploy configs.
- **Classwork** (`classwork.md` + the files above) — three exercises: run `config.py` under no env vars / shell overrides / a `.env` file and confirm the precedence order, start `cloud_ready_api.py` on an env-provided port and prove graceful shutdown live under `SIGTERM`, then run `cloud_readiness_check.py` against both example apps and read `render.yaml`/`fly.toml`/`Procfile` side by side for how each injects `PORT` and health-checks `/health`.
- **Homework** (`homework/homework.md`, ~2 hours) — finish the `max_upload_mb` stretch (reject oversized uploads with `413`), finish the linter's 7th check (flag `debug=True`), write the conceptual answer on shell-form vs. exec-array-form `CMD`, optionally deploy the real image to a free-tier cloud platform at home, and reread `formulas.md` §1-4 before committing.

All three sources point at the same two code deliverables — the `max_upload_mb` stretch in `config.py`/`cloud_ready_api.py` and the `debug=True` check in `cloud_readiness_check.py` — plus a short written answer on `CMD` shell-form vs. exec-array-form that homework #3 calls "the actual point of today, not a formality.\"

## Assignment — Classwork exercises

### Exercise 1 — `config.py` + `cloud_ready_api.py` (~25 min)

**What's being asked:** confirm `config.py`'s settings change correctly across three sources (defaults, shell env vars, `.env` file) with zero code changes, then start `cloud_ready_api.py` on an env-provided port and prove graceful shutdown live.

**Approach:**
1. `pip install -r requirements.txt --break-system-packages`, `cp .env.example .env`.
2. Run `python3 config.py` with no env vars set, then `PORT=10000 ENVIRONMENT=production LOG_FORMAT=json python3 config.py`, and confirm the printed JSON changes correctly.
3. Edit `.env` to set `PORT=9090`, then run `env -i PATH=$PATH python3 config.py` (zero shell vars) and confirm `config.py` picks up the `.env` value instead of the hardcoded default.
4. `PORT=8500 LOG_FORMAT=json python3 cloud_ready_api.py &`, then `curl http://127.0.0.1:8500/health`; confirm it is reachable on 8500 (env-provided) and `curl http://127.0.0.1:8000/health` is refused (not the code's hardcoded default).
5. Submit a job with `curl -X POST http://127.0.0.1:8500/classify-async -F "file=@sample_images/digit_0_label_0.png"`, then immediately send `SIGTERM` (Ctrl+C or `kill <pid>`) while the job is still running; read the logs for "Waiting for background tasks to complete" followed by the job actually finishing before the process exits.
6. **Stretch (= homework #1):** add `max_upload_mb: int = 5` to `Settings`, check `len(raw_bytes)` against `settings.max_upload_mb * 1024 * 1024` in `/classify-async`, and return `413` when it's exceeded; test with an oversized file (expect `413`) and a normal-sized image (expect success).

### Exercise 2 — `cloud_readiness_check.py` (~20 min)

**What's being asked:** confirm the linter reports clean on the good API file and 3 errors + 3 warnings on the deliberately bad one, fix the bad file one issue at a time until it passes, and add a 7th check.

**Approach:**
1. `python3 cloud_readiness_check.py cloud_ready_api.py` (expect "no issues found", exit 0) and `python3 cloud_readiness_check.py app_bad_example.py` (expect 3 errors + 3 warnings, exit 1).
2. Open `app_bad_example.py`; fix each flagged line one at a time — the hardcoded `127.0.0.1` bind, the hardcoded `port=8000` in `uvicorn.run()`, the missing `/health` route, the bare `os.environ.get(...)` calls, the hardcoded `API_KEY = "sk-..."`, and the `FileHandler`-style logging — re-running the linter after each fix, noting which change resolved which reported issue.
3. **Stretch (= homework #2):** add a 7th check to `check_source()` matching `debug\s*=\s*True` inside a `FastAPI(...)` or `uvicorn.run(...)` call; write a small test app with that mistake, confirm the new check fires, then re-run against `cloud_ready_api.py` and `app_bad_example.py` to confirm the existing 6 checks still behave the same.

### Exercise 3 — Deploy configs (~20 min)

**What's being asked:** read `render.yaml`, `fly.toml`, and `Procfile` side by side, identify how each injects `PORT` and health-checks `/health`, re-run the syntax checks, and explain which one command sets `REDIS_URL` without it ever appearing in source.

**Approach:**
1. Read the three files together: `render.yaml`'s `envVars` + `fromService` block wires `REDIS_URL` from the attached Redis blueprint service and `healthCheckPath: /health`; `fly.toml`'s `[http_service.checks]` block points at `path = "/health"` while Fly injects `PORT` itself via `[http_service]`; `Procfile`'s `web: uvicorn ... --port $PORT` line is what any Heroku-style platform reads to both start the process and supply `$PORT`.
2. Re-run the syntax checks: `python3 -c "import yaml; d = yaml.safe_load(open('render.yaml')); print(d['services'][0]['healthCheckPath'])"` and `python3 -c "import toml; d = toml.load('fly.toml'); print(d['http_service']['checks'][0]['path'])"`.
3. Read `cloud_deploy_commands.md`; the single command where `REDIS_URL` gets set without ever appearing in source is `fly secrets set REDIS_URL=<the-connection-string-fly-gives-you>` (or Render's equivalent `fromService` wiring, or Heroku's `heroku addons:create heroku-redis:mini`) — each injects the URL as a runtime secret rather than a committed value.
4. **At home (stretch, needs a free-tier cloud account):** pick one platform, follow `cloud_deploy_commands.md`'s real steps, deploy the image, and hit `/health` from a phone off home wifi to confirm public reachability.

## Assignment — Homework (~2 hours self-study)

**1. Finish the config stretch** — *What's asked:* add `max_upload_mb: int = 5` to `Settings`, and in `/classify-async` check the uploaded file's byte length against `settings.max_upload_mb * 1024 * 1024`, returning `413` when it's exceeded; test with a real oversized file and confirm a normal-sized image still succeeds. *Approach:* read `raw_bytes` before passing it to the image decoder, compare its length against the computed byte threshold first, raise `HTTPException(status_code=413, ...)` on failure, then test both paths with curl.

**2. Finish the linter stretch** — *What's asked:* add a 7th check to `cloud_readiness_check.py` flagging `debug=True` passed to `FastAPI(...)` or `uvicorn.run(...)`; write a small test app with that mistake, confirm the new check catches it, and re-run against `cloud_ready_api.py` and `app_bad_example.py` to confirm the existing 6 checks still pass. *Approach:* add a regex matching `debug\s*=\s*True` scoped to lines containing `FastAPI(` or `uvicorn.run(`, append a `warning` `Issue` at that line, and verify in isolation before the full regression run.

**3. The conceptual question (~3-4 sentences)** — *What's asked:* explain why a Dockerfile's `CMD` needs the shell form (`CMD uvicorn ... --port ${PORT}`) rather than the exec-array form (`CMD ["uvicorn", ..., "--port", "8000"]`) to pick up a cloud-platform-assigned port, specifically what differs about WHEN each form reads `PORT`. *Approach to answering:* the exec-array form is executed directly by the container runtime with no shell involved, so `${PORT}` is never substituted and is passed through as the literal string `${PORT}`; the shell form is handed to `/bin/sh -c`, which performs variable expansion at the moment the container actually starts — meaning the exec-array form's port value is frozen in at image BUILD time (whatever was hardcoded), while the shell form's port value is read fresh at container START time, which is the only point at which the cloud platform has actually assigned and injected `PORT`.

**4. Optional — at home, with a free-tier cloud account (not graded)** — *What's asked:* deploy the Day 51 image to Render, Fly.io, or a Heroku-style PaaS, confirm `/health` is reachable from a phone off home wifi, submit and poll a real classify job through the deployed URL, then redeploy once and try to spot the old container's graceful-shutdown log line before the new one takes over. *Approach:* follow `cloud_deploy_commands.md`'s steps for the chosen platform exactly, and bring notes on what worked (or didn't, and why) to the next class.

**5. Reread + commit** — *What's asked:* reread `formulas.md` sections 1-4 (config via environment, `$PORT`, structured logging, graceful shutdown) and commit the `max_upload_mb` stretch, the `debug=True` linter check, and the written answer to #3 to the course repo. *Approach:* read the four sections again with the day's actual code in front of me, then `git add . && git commit -m "Day 51 — env-based config, cloud-readiness linter, and the CMD shell-vs-exec-array answer"`.